In [ ]:
# 3D Block World Reconstruction from 2D Images

# Phase 1: Environment Setup & Ground Plane Segmentation
# In this phase, we load the target image and convert it into the HSV color space to isolate the ground plane from the objects. 
# We apply a saturation threshold to segment the background effectively.

In [1]:
# --- Core Libraries ---
import scipy.sparse.linalg as spl
from scipy.sparse import coo_matrix
import cv2
import numpy as np
import plotly.express as px
import plotly.graph_objs as go
import time

# --- Image Loading ---
image_data = cv2.imread("./image2.png", cv2.IMREAD_GRAYSCALE)
color_image_data = cv2.imread("./image2.png")

print(f"Original Image Shape: {color_image_data.shape}")

# --- Ground Plane Segmentation ---
hsv = cv2.cvtColor(color_image_data, cv2.COLOR_BGR2HSV)

GREY_SATURATION_TRESHOLD = 80

lower_bound = np.array([0, 0, 0])
upper_bound = np.array([179, GREY_SATURATION_TRESHOLD, 255])

mask = cv2.inRange(hsv, lower_bound, upper_bound)

ground_plane = cv2.bitwise_and(color_image_data, color_image_data, mask=mask)

height, width = image_data.shape[:2]

# Refine ground plane mask
mask = np.mean(ground_plane, axis=2) > 80
ground_plane[~mask] = 0

# Visualize segmented ground plane
ground_plot = px.imshow(ground_plane)
ground_plot.update_layout(title="Segmented Ground Plane")
ground_plot.show()

In [ ]:
# Phase 2: Sobel Gradients & Edge Classification
# We compute the image gradients using the Sobel operator to find edge magnitudes and directions. 
# Based on the gradient angles, we classify edges into three categories: vertical edges, horizontal edges, and 
# contact edges (where horizontal edges touch the ground plane).

In [7]:
# --- Sobel Gradients Computation ---
sobelx = cv2.Sobel(image_data, cv2.CV_64F, 1, 0, ksize=5)
sobely = cv2.Sobel(image_data, cv2.CV_64F, 0, 1, ksize=5)

gradient_matrix = np.stack([sobelx, sobely], axis=-1)

gradient_mag = np.sqrt(sobely**2 + sobelx**2)
magnitude, angles = cv2.cartToPolar(sobelx, sobely, angleInDegrees=True)

# Define ground plane boolean mask
is_ground_plane = np.where(np.all(ground_plane != 0), 1, 0)

output = image_data.copy()

# --- Edge Classification Parameters ---
EDGE_THRESHOLD = 800
STEP = 10
PHI = 22
ONE_OVER_COS_PHI = 1 / np.cos(np.deg2rad(PHI))
ANGLE_THRESHOLD = 15
SIN_ANGLE_THRESHOLD = np.sin(np.deg2rad(ANGLE_THRESHOLD))
max_magnitude = np.max(magnitude)

# --- Edge Masks Definition ---
edges = np.where((magnitude > EDGE_THRESHOLD) & ~is_ground_plane, 1, 0)

vertical_edge_mask = np.where(np.abs(np.sin(np.deg2rad(angles))) < SIN_ANGLE_THRESHOLD, 1, 0)
vertical_edges = edges & vertical_edge_mask

horizontal_edges = edges & ~vertical_edge_mask

# Identify contact edges (intersection between horizontal edges and the ground plane)
contact_edges = np.zeros_like(horizontal_edges)
for y in range(height-10):
    for x in range(width):
        if horizontal_edges[y, x] != 0:
            if np.all(ground_plane[y+10, x] != 0) and np.all(ground_plane[y - 10, x] == 0):
                contact_edges[y, x] = horizontal_edges[y, x]

# Visualize classified edges
combined_edges = np.where(vertical_edges, 1, np.where(contact_edges, 2, np.where(horizontal_edges, 3, 0)))
px.imshow(combined_edges).update_layout(title="Classified Edges (1: Vert, 2: Contact, 3: Horiz)").show()

# --- Edge Orientations Plotting ---
fig = px.imshow(color_image_data)
fig.update_layout(
    xaxis_range=[0, width],
    yaxis_range=[0, height],
    yaxis_autorange="reversed",
    title="Edge Orientations (Gradient Vectors)",
)

arrows = []
for y in range(0, height, STEP):
    for x in range(0, width, STEP):
        if not edges[y, x]:
            continue
        rad = np.deg2rad(angles[y, x])
        normalized_length = 40
        x2 = int(x + normalized_length * np.cos(rad))
        y2 = int(y + normalized_length * np.sin(rad))

        arrows.append(go.layout.Annotation(
            xref="x", yref="y", x=x2, y=y2,
            ax=x, ay=y, text="", showarrow=True,
            arrowwidth=1, arrowcolor="red", arrowhead=3,
            axref="x", ayref="y",
        ))
fig.update_layout(annotations=arrows)
fig.show()

In [ ]:
## Phase 3: Sparse System Construction
# To reconstruct the Y-coordinate, we build a massive system of linear equations based on spatial constraints:
# * **Vertical Edges:** Derived mathematically based on the camera tilt angle $Y_{y} = \frac{1}{\cos(\phi)}$
# * **Horizontal Edges:** Orthogonal gradient constraints.
# * **Contact Edges:** Ground zero constraints $Y = 0$.
# * **Faces:** Second-order derivatives (Laplacians) set to zero to enforce planar surfaces.

# Due to the massive dimensionality (e.g., >48000x48000), we utilize a Sparse Coordinate Matrix (`coo_matrix`).

In [13]:
full_length = width * height

constraint_data = []
constraint_row_indices = []
constraint_column_indices = []
b = []

def add_constraint_coefficient(condition_index: int, x: int, y: int, value: int):
    """Helper function to populate sparse matrix arrays."""
    flat_index = y * width + x
    constraint_row_indices.append(condition_index)
    constraint_column_indices.append(flat_index)
    constraint_data.append(value)

current_condition_index = 0

# --- Building the Constraints System ---
for y in range(0, height):
    for x in range(0, width):

        # Skip explicit ground plane pixels
        if np.all(ground_plane[y, x] != 0):
            continue

        # Constraint 1: Vertical Edges
        if vertical_edges[y, x]:
            add_constraint_coefficient(current_condition_index, x, y+1, 1)
            add_constraint_coefficient(current_condition_index, x, y-1, -1)
            b.append(-ONE_OVER_COS_PHI)
            current_condition_index += 1
            continue
        
        # Constraint 2 & 3: Horizontal Edges
        elif horizontal_edges[y, x]:
            # Contact edges (Y = 0)
            if contact_edges[y, x]:
                add_constraint_coefficient(current_condition_index, x, y, 1)
                b.append(0)
                current_condition_index += 1
                continue

            # Standard horizontal edges (Orthogonal vector constraint)
            mag = gradient_mag[y, x]
            n_x, n_y = gradient_matrix[y, x] / mag

            add_constraint_coefficient(current_condition_index, x + 1, y, -n_y)
            add_constraint_coefficient(current_condition_index, x - 1, y, n_y)
            add_constraint_coefficient(current_condition_index, x, y + 1, n_x)
            add_constraint_coefficient(current_condition_index, x, y - 1, -n_x)

            b.append(0)
            current_condition_index += 1
            continue
            
        # Constraint 4: Planar Faces (Second derivatives = 0)
        else:
            add_constraint_coefficient(current_condition_index, x, y, -2)
            add_constraint_coefficient(current_condition_index, x + 2, y, 1)
            add_constraint_coefficient(current_condition_index, x - 2, y, 1)
            b.append(0)
            current_condition_index += 1

            add_constraint_coefficient(current_condition_index, x, y, -2)
            add_constraint_coefficient(current_condition_index, x, y + 2, 1)
            add_constraint_coefficient(current_condition_index, x, y - 2, 1)
            b.append(0)
            current_condition_index += 1

            add_constraint_coefficient(current_condition_index, x + 1, y + 1, 1)
            add_constraint_coefficient(current_condition_index, x - 1, y + 1, -1)
            add_constraint_coefficient(current_condition_index, x + 1, y - 1, -1)
            add_constraint_coefficient(current_condition_index, x - 1, y - 1, 1)
            b.append(0)
            current_condition_index += 1

print(f"Total Constraints Generated: {current_condition_index}")
print(f"B vector length: {len(b)}")

# Construct Sparse Matrix
constraint_matrix = coo_matrix(
    (constraint_data, (constraint_row_indices, constraint_column_indices)),
    shape=(current_condition_index, height * width)
)

constraint_coefficients = np.array(b)

print(f"Constraint Matrix Shape: {constraint_matrix.shape}")
print(f"Constraint Coefficients Shape: {constraint_coefficients.shape}")

In [ ]:
## Phase 4: System Resolution & 3D Interactive Visualization
# We solve the highly sparse linear system utilizing `scipy.sparse.linalg.lsqr` to approximate the 3D depth map (Y coordinates). 
# Finally, we map the original 2D image texture onto the reconstructed 3D surface using Plotly.

In [19]:
# --- Solving the Sparse Linear System ---
print("Executing LSQR Solver...")
Y_flat, *_ = spl.lsqr(constraint_matrix.tocsr(), constraint_coefficients)

# Reshape back to image dimensions
Y_solved = Y_flat.reshape(height, width)
np.save("Y_solved.npy", Y_solved)
print("System solved successfully. Projection map saved.")

# --- Interactive 3D Plotting ---
try:
    img = cv2.imread("./image2.png")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    texture = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
except Exception:
    texture = Y_solved

x = np.arange(0, width)
y = np.arange(0, height)
X, Y = np.meshgrid(x, y)

# Generate Interactive Surface Plot
fig = go.Figure(data=[go.Surface(
    z=Y_solved, 
    x=X, 
    y=Y,
    surfacecolor=texture,
    colorscale='Viridis',
    colorbar=dict(title="Height Map")
)])

fig.update_layout(
    title='3D Object Reconstruction',
    scene=dict(
        xaxis=dict(title='X'),
        yaxis=dict(title='Y', autorange="reversed"),
        zaxis=dict(title='Height (Z)', range=[np.min(Y_solved), np.max(Y_solved) * 1.5]),
        aspectratio=dict(x=1, y=height/width, z=0.2)
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()